# 01 · The Elastic Dislocation Forward Model

*How slip on a buried fault produces surface displacement.*

**Learning objectives**

- explain the elastic half-space and dislocation assumptions
- translate strike, dip, rake, depth, and slip into a forward model
- verify the linear map $\mathbf d=G\mathbf m$ two ways
- compute scalar moment and moment magnitude

**Prerequisites:** Chapter 00 or equivalent NumPy familiarity; [conventions](../docs/conventions.md)  
**Estimated time:** 60–90 minutes

Notation follows the [glossary](../docs/glossary.md); axes, units, signs, and
ordering follow [GeoDef conventions](../docs/conventions.md).


## Motivation

Surface motion is the observable footprint of slip that occurs kilometers
below us. The forward problem asks a physically constrained question: *if the
fault geometry and slip were known, what would GNSS or InSAR measure?* That
question must be understood before its inverse can be trusted.

GeoDef uses a homogeneous, isotropic, linear-elastic half-space. The free
surface is flat; material parameters do not vary with position; deformation is
recoverable and proportional to load. These assumptions make the Okada
solution analytic and fast. They also exclude topography, layering, poroelastic
response, and permanent off-fault deformation—limitations revisited in
Chapter 13.


## Theory deepening: dislocations, geometry, and linearity

A **dislocation** is a displacement discontinuity across an internal surface.
Volterra introduced the construction; Steketee connected it to faults; Okada
derived closed-form displacement and derivative fields for rectangular sources
in an elastic half-space. GeoDef supplies source geometry and in-plane slip and
returns East, North, and Up displacement.

For fixed geometry and elastic medium, superposition gives

$$\mathbf d = G\mathbf m+\boldsymbol\epsilon,$$

where the first $N$ entries of $\mathbf m$ are strike slip and the final $N$
are dip slip. Doubling $\mathbf m$ doubles the noise-free displacement. `mu`
controls scalar moment while `nu` changes the displacement kernels; both live
on `ElasticMedium` so they are declared together.

Strike is clockwise from north, dip is down from horizontal, and rake is
within the fault plane. Under GeoDef's convention, rake $90^\circ$ is reverse
slip. See the [glossary](../docs/glossary.md) for each symbol and the
[conventions](../docs/conventions.md) for signs and units.


## The forward and inverse problems

Two complementary questions sit at the heart of geodetic fault modeling:

- **Forward problem** — *given* model parameters (slip on the fault), predict
  the data (surface displacements). **This notebook.**
- **Inverse problem** — *given* data, estimate the model parameters that best
  explain them. Notebooks 03 onward.

Both are tied together by a single linear relationship:

$$ \mathbf{d} = G\,\mathbf{m} + \mathbf{e} $$

| symbol | meaning | shape |
| --- | --- | --- |
| $\mathbf{d}$ | observations (surface displacements) | $(N_\text{obs},)$ |
| $\mathbf{m}$ | model parameters (slip on each patch) | $(2N,)$ |
| $G$ | Green's function matrix (the physics) | $(N_\text{obs},\, 2N)$ |
| $\mathbf{e}$ | measurement error | $(N_\text{obs},)$ |

The matrix $G$ encodes the elastic physics: column $j$ is the surface
displacement produced by *one unit of slip* on patch $j$. `geodef` builds it
from the Okada (1985) analytical solution for a rectangular dislocation in an
elastic half-space. Notebook 02 opens up $G$ in detail; here we treat it as the
engine behind the forward model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import geodef

# reload geodef automatically if you edit the source while experimenting
%load_ext autoreload
%autoreload 2

## 1. Build a discretized fault

We model a subduction-zone megathrust: a shallow-dipping rectangular fault,
180 km long and 90 km wide, with its center 25 km deep. We split it into a grid
of `n_length × n_width = 12 × 6 = 72` patches so that slip can vary across the
fault surface.

`Fault.planar()` takes the centroid location, orientation (strike, dip), size,
and the patch counts. **All distances are in meters; all angles in degrees.**

> **Coordinate conventions.** Geographic inputs are latitude, longitude, and
> depth (positive down, meters). Strike is degrees clockwise from north; dip is
> degrees down from horizontal. Internally `geodef` works in a local
> East/North/Up Cartesian frame and converts at the boundaries.

In [ ]:
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,   # centroid 25 km deep
    strike=315.0, dip=15.0,            # NW-striking, shallow dip
    length=180e3, width=90e3,          # 180 km x 90 km
    n_length=12, n_width=6,            # -> 72 patches
)

nL, nW = fault.grid_shape
print(f"Engine:      {fault.engine}")
print(f"Patches:     {fault.n_patches}  ({nL} along-strike x {nW} down-dip)")
print(f"Patch area:  {fault.areas[0] / 1e6:.0f} km^2 each")
print(f"Depth range: {fault.centers_geo[:, 2].min() / 1e3:.1f} - "
      f"{fault.centers_geo[:, 2].max() / 1e3:.1f} km")

The fault is a dipping plane below the surface. Let's look at it in 3-D, colored
by depth. `geodef.plot.fault3d` returns a Matplotlib 3-D axes, so you can keep
customizing it after the call.

In [ ]:
ax=geodef.plot.fault3d(
    fault, color_by="depth", view=(32, -75),
    title="Megathrust geometry (72 patches)",
)
plt.show()

## 2. Choose a slip distribution

Slip on each patch has two components in the fault plane:

- **strike-slip** — horizontal motion along strike,
- **dip-slip** — motion up or down the dip direction (thrust or normal faulting).

`geodef` stores the full slip model as a single **blocked** vector of length
$2N$ — all strike-slip values first, then all dip-slip values:

$$ \mathbf{m} = [\,\underbrace{s^\text{ss}_0, \dots, s^\text{ss}_{N-1}}_{\text{strike-slip}},\ \underbrace{s^\text{ds}_0, \dots, s^\text{ds}_{N-1}}_{\text{dip-slip}}\,] $$

A megathrust earthquake is almost pure thrust, so we place a smooth **dip-slip**
asperity near the center of the fault and leave strike-slip at zero. Patches are
numbered with the along-strike index varying fastest, so patch `k = j*nL + i`
sits at strike index `i` and dip index `j`.

In [ ]:
# patch grid indices
i = np.arange(fault.n_patches) % nL      # along-strike index
j = np.arange(fault.n_patches) // nL     # down-dip index

# smooth Gaussian asperity of thrust (dip-slip), peaking near the center
i0, j0 = (nL - 1) / 2, nW / 2
slip_dip = 3.0 * np.exp(-(((i - i0) / 3.0) ** 2 + ((j - j0) / 1.5) ** 2))
slip_strike = np.zeros(fault.n_patches)

print(f"Peak dip-slip: {slip_dip.max():.2f} m")

The input slip distribution, shown in the fault's own along-strike /
along-dip frame with the up-dip edge at the top (the default for
`plot.slip`); a black line marks the up-dip edge:

In [ ]:
geodef.plot.slip(
    fault, slip_dip, cmap="YlOrRd",
    colorbar_label="Dip-slip (m)", title="Input slip distribution",
)
plt.show()

## 3. Predict surface displacement

Now the forward model. We place a grid of synthetic GNSS stations at the surface
and ask: *how does the ground move?*

We'll do it **twice** — first spelling out the matrix algebra, then with the
one-line `geodef` helper — to show they are exactly the same computation.

> **Watch the argument order.** Data classes such as `GNSS` take **longitude
> first, then latitude** (`GNSS(lon=..., lat=..., ...)`); the constructors are keyword-only so
the order can never be silently confused.
> The fault methods take **latitude first** (`fault.displacement(lat, lon, ...)`).

In [ ]:
# grid of surface stations (build the arrays in lon, lat order)
sta_lon, sta_lat = np.meshgrid(
    np.linspace(98.5, 101.5, 8),
    np.linspace(-3.6, -0.4, 8),
)
sta_lon, sta_lat = sta_lon.ravel(), sta_lat.ravel()
print(f"{sta_lat.size} stations")

### 3a. The explicit calculation: $\mathbf{d} = G\mathbf{m}$

`fault.greens_matrix()` builds $G$ for these stations. Its shape is $(3M, 2N)$:
three displacement components (E, N, U) at each of the $M$ stations in the rows,
and the $2N$ blocked slip parameters in the columns. We form the blocked slip
vector $\mathbf{m}$, multiply, then unpack the interleaved
`[e, n, u, e, n, u, ...]` result.

In [ ]:
G = fault.greens_matrix(sta_lat, sta_lon)        # (3M, 2N)
m = np.concatenate([slip_strike, slip_dip])      # blocked [ss | ds], length 2N

d = G @ m                                         # the forward model
pred_e, pred_n, pred_u = d[0::3], d[1::3], d[2::3]

print(f"G shape: {G.shape}  ->  (3 x {sta_lat.size} stations, "
      f"2 x {fault.n_patches} patches)")
print(f"m shape: {m.shape}")
print(f"Max horizontal displacement: {np.hypot(pred_e, pred_n).max():.3f} m")
print(f"Max vertical displacement:   {np.abs(pred_u).max():.3f} m")

### 3b. The one-liner: `fault.displacement()`

`geodef` wraps exactly the steps above. `fault.displacement()` builds $G$, forms
the blocked slip vector, multiplies, and returns the unpacked components — so the
result is identical to what we just computed by hand.

In [ ]:
ue, un, uz = fault.displacement(
    sta_lat, sta_lon,
    slip_strike=slip_strike, slip_dip=slip_dip,
)

print("Identical to G @ m:",
      np.allclose([ue, un, uz], [pred_e, pred_n, pred_u]))

## 4. Visualize the predicted displacements

To plot vectors we wrap the predictions in a `GNSS` object — the same container
you would use for real data. We draw the fault footprint with `plot.map_view` for
spatial context, then overlay the displacement field with `plot.vectors`:
horizontal arrows plus the vertical component as colored dots (with a colorbar).
Everything is in the local East/North frame centered on the fault, and the arrows
are drawn on top of the dots so they stay visible even where the dots are large.

In [ ]:
n_sta = sta_lat.size
gnss = geodef.data.gnss(
    lon=sta_lon, lat=sta_lat,                  # lon, lat
    east=pred_e, north=pred_n, up=pred_u,            # E, N, U displacements
    sigma_east=np.full(n_sta, 0.002),          # nominal uncertainties
    sigma_north=np.full(n_sta, 0.002),
    sigma_up=np.full(n_sta, 0.005),
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
geodef.plot.slip(fault, slip_dip, ax=axes[0], cmap="YlOrRd",
                 coords="geographic",
                 colorbar_label="Dip-slip (m)", title="Slip (model)")

# fault footprint for context, then the displacement field on top
geodef.plot.map_view(fault, ax=axes[1])
scale = 35.0 / np.hypot(pred_e, pred_n).max()   # arrows ~35 km at peak
geodef.plot.vectors(gnss, fault, ax=axes[1], components="both", scale=scale,
                    legend=True, scale_arrow=0.2, scale_arrow_label="0.2 m",
                    title="Predicted surface displacement")
plt.show()

The horizontal arrows point toward the trench-ward, up-dip edge and the vertical
field shows uplift above the slip patch flanked by subsidence — the classic
signature of a buried thrust.

## 5. How big an earthquake is this?

The seismic moment is $M_0 = \mu \sum_k s_k A_k$ (shear modulus $\times$ slip
$\times$ area, summed over patches), and the moment magnitude follows from
$M_w = \tfrac{2}{3}\log_{10} M_0 - 6.07$. `geodef` computes both (default shear
modulus $\mu = 30$ GPa).

In [ ]:
Mw = fault.magnitude(slip_dip)
M0 = fault.moment(slip_dip)
print(f"Seismic moment M0: {M0:.2e} N m")
print(f"Moment magnitude:  Mw {Mw:.2f}")

## Checkpoint questions

**Why does doubling slip double displacement but not $M_w$?**

<details><summary>Answer</summary>



The elastic displacement is linear in slip. Moment is also linear, but magnitude is logarithmic in moment.



</details>

**What changes when `mu` changes?**

<details><summary>Answer</summary>



Scalar moment changes directly. In the current half-space kernels, displacement depends on Poisson's ratio rather than shear modulus.



</details>

**Why retain a blocked vector at all?**

<details><summary>Answer</summary>



It is the natural column ordering of the linear system; named functions and result views protect routine interpretation.



</details>


## Common mistakes

- Passing kilometers where meters are required makes a source 1,000 times too shallow or small.
- Confusing centroid depth with top-edge depth shifts the entire source.
- A smooth displacement field does not imply smooth or well-resolved slip.


## Recap

- Fixed geometry makes surface displacement linear in slip.
- Strike/dip/rake describe geometry and in-plane motion; ENU describes observations.
- Moment depends on rigidity, area, and slip magnitude, while magnitude is logarithmic.

Return to the [workflow decision guides](../docs/workflow.md) when adapting
this method. The course map in [the tutorial README](README.md) identifies the
next chapter and optional branches.


## Exercises

Predict the qualitative outcome before running each experiment. Worked
solutions are published separately in `solutions/01_forward_model_solutions.ipynb`.

1. Change dip to 15° and 60°. Predict the vertical/horizontal balance before running.
2. Move the centroid 10 km deeper and compare peak displacement.
3. Replace dip slip with pure strike slip and interpret the quadrant pattern.
4. Challenge: sum predictions from two slip distributions and verify exact superposition.


## Further reading

- Okada (1985), rectangular dislocations in an elastic half-space.
- Hanks & Kanamori (1979), the moment-magnitude scale.
- Segall (2010), geodetic earthquake-cycle physics.
